# 05 · Data: Dataset, DataLoader, padding, masks and packing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/pytorch/05_data_pipeline.ipynb) &nbsp; 📖 Concept deck: [Data pipelines for LLMs](https://nilesh-salpe.github.io/visual-ai/pytorch/05-data/) &nbsp; ⏱️ Suggested time: **30 min**

**You will build:** a token-window `Dataset`, a collate function that pads variable-length examples and builds labels with `ignore_index=-100`, and sequence packing that removes padding waste.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)
print("PyTorch", torch.__version__)

from torch.utils.data import Dataset, DataLoader
PAD = 0
DOCS = [[5, 6, 7], [8, 9], [10, 11, 12, 13, 14], [15], [16, 17, 18, 19]]   # token ids of five short documents

## Task 1: Fixed-length windows over a token stream

Pretraining usually concatenates documents into one long stream and cuts windows of `T + 1` tokens: inputs are the first T, targets the last T (shifted by one). Implement `__len__` and `__getitem__(i)` returning `(x, y)` long tensors for non-overlapping windows.

In [ ]:
class WindowDataset(Dataset):
    def __init__(self, tokens, T):
        self.tokens, self.T = torch.as_tensor(tokens), T
    def __len__(self):
        # TODO
        raise NotImplementedError
    def __getitem__(self, i):
        # TODO
        raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_WindowDataset(cls):
    ds = cls(list(range(100, 121)), 4)
    assert len(ds) == 5, "21 tokens → 5 full windows of 4 inputs + 1 target"
    x, y = ds[1]
    assert x.tolist() == [104, 105, 106, 107] and y.tolist() == [105, 106, 107, 108]
    xb, yb = next(iter(DataLoader(ds, batch_size=2)))
    assert xb.shape == (2, 4) and xb.dtype == torch.long

check("WindowDataset", _test_WindowDataset, WindowDataset)

## Task 2: Pad, mask and label

For fine-tuning, each example is its own sequence. Write `pad_collate(batch)` for a list of 1-D token lists: right-pad inputs with `PAD`, build `attention_mask` (1 for real tokens), and `labels` = next tokens with **−100** at padding and at the last real position, so `F.cross_entropy(…, ignore_index=-100)` (the default) skips them.
Return `(input_ids, attention_mask, labels)`, all shape (B, L) with L the longest length.

In [ ]:
def pad_collate(batch):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_pad_collate(f):
    ids, mask, lab = f([[5, 6, 7], [8, 9]])
    assert ids.tolist() == [[5, 6, 7], [8, 9, 0]]
    assert mask.tolist() == [[1, 1, 1], [1, 1, 0]]
    assert lab.tolist() == [[6, 7, -100], [9, -100, -100]]
    logits = torch.randn(2, 3, 20)
    full = F.cross_entropy(logits.view(-1, 20), lab.view(-1))
    keep = lab.view(-1) != -100
    assert torch.allclose(full, F.cross_entropy(logits.view(-1, 20)[keep], lab.view(-1)[keep])), "-100 positions are ignored"

check("pad_collate", _test_pad_collate, pad_collate)

## Task 3: Pack documents into fixed rows

Padding wastes compute. **Packing** concatenates documents (each followed by an end-of-text id `eot`) into rows of exactly `L` tokens. Return `(rows, doc_ids)`: a (n_rows, L) tensor with the stream cut into rows (drop the incomplete tail) and a same-shaped tensor saying which document each token came from (so attention can be restricted to the same document).

In [ ]:
def pack(docs, L, eot=1):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_pack(f):
    rows, did = f(DOCS, 6)
    assert rows.shape == did.shape and rows.shape[1] == 6
    stream = sum(([*d, 1] for d in DOCS), [])
    assert rows.flatten().tolist() == stream[:rows.numel()], "rows are the concatenated stream"
    assert rows.shape[0] == len(stream) // 6
    assert did[0].tolist() == [0, 0, 0, 0, 1, 1], "first row: doc 0 (3 tokens + eot) then doc 1"

check("pack", _test_pack, pack)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Fixed-length windows over a token stream
class WindowDataset_solution(Dataset):
    def __init__(self, tokens, T):
        self.tokens, self.T = torch.as_tensor(tokens), T
    def __len__(self):
        return (len(self.tokens) - 1) // self.T
    def __getitem__(self, i):
        s = i * self.T
        chunk = self.tokens[s:s + self.T + 1]
        return chunk[:-1].long(), chunk[1:].long()

check("WindowDataset (reference)", _test_WindowDataset, WindowDataset_solution)

In [ ]:
# @title Solution for Task 2: Pad, mask and label
def pad_collate_solution(batch):
    L = max(len(b) for b in batch)
    ids = torch.full((len(batch), L), PAD, dtype=torch.long)
    mask = torch.zeros(len(batch), L, dtype=torch.long)
    lab = torch.full((len(batch), L), -100, dtype=torch.long)
    for i, b in enumerate(batch):
        t = torch.tensor(b); ids[i, :len(b)] = t; mask[i, :len(b)] = 1; lab[i, :len(b) - 1] = t[1:]
    return ids, mask, lab

check("pad_collate (reference)", _test_pad_collate, pad_collate_solution)

In [ ]:
# @title Solution for Task 3: Pack documents into fixed rows
def pack_solution(docs, L, eot=1):
    stream, owner = [], []
    for i, d in enumerate(docs):
        stream += [*d, eot]; owner += [i] * (len(d) + 1)
    n = len(stream) // L
    return torch.tensor(stream[:n * L]).view(n, L), torch.tensor(owner[:n * L]).view(n, L)

check("pack (reference)", _test_pack, pack_solution)

---
## Experiments
**How much does padding waste?** Compare the share of padding tokens when batching documents of varied lengths with padding vs packing, and build the block-diagonal mask that keeps packed documents from attending to each other.

In [ ]:
g = torch.Generator().manual_seed(0); lengths = torch.randint(20, 400, (64,), generator=g).tolist()
for bs in [8, 32]:
    pad = sum(bs * max(lengths[i:i + bs]) - sum(lengths[i:i + bs]) for i in range(0, 64, bs)); total = sum(lengths) + pad
    print(f"batch {bs:2d}: {pad / total:.0%} of computed tokens are padding")
rows, did = pack_solution(DOCS, 6)
same = did[0][:, None] == did[0][None, :]; causal = torch.tril(torch.ones(6, 6, dtype=torch.bool))
print("document-causal mask for row 0:\n", (same & causal).int())

---
## Takeaways
* `Dataset` returns one example; `DataLoader` batches, shuffles and (with workers) prefetches; `collate_fn` controls how examples become a batch.
* Labels use −100 (cross-entropy's default `ignore_index`) for padding and prompt tokens you don't want to train on.
* Packing removes padding waste; a document mask (or per-document position ids) keeps packed documents independent.